# SOXX 2019 旧表单至 N-PORT CUSIP 身份桥：报告期修复

保留2019-06缺口；只读一份原始旧表单和一个原生 Qlib Recorder 成员对象，不计算换手或收益。

In [1]:
import os,sys,json,hashlib,re,unicodedata,collections
from pathlib import Path
import qlib
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root));card=root/'research/stage1_soxx_basket_rotation_2019_name_cusip_bridge_v2_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];assert plan['frozen'] and not (card/'run_state.json').exists();repair=json.loads((card/'parser_repair_plan.json').read_text());assert repair['frozen'] and repair['parent_plan_sha256']==hashlib.sha256((card/'plan.json').read_bytes()).hexdigest();sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['parent_failed_v1']['card']/'failure.json')==plan['parent_failed_v1']['failure_sha256']
legacy=root/src['legacy_2019_transition_result'];modern=root/src['modern_card'];assert sha(legacy)==src['legacy_result_sha256'] and sha(modern/'summary.json')==src['modern_summary_sha256'] and sha(root/'research/stage1_soxx_reported_basket_rotation_footnote_repair_20261023/result.json')==src['legacy_footnote_repair_result_sha256']
summary=json.loads((modern/'summary.json').read_text());artifact=modern/'qlib_recorder'/summary['experiment_id']/summary['recorder_id']/'artifacts/analysis/member_rows';assert sha(artifact)==src['modern_member_rows_artifact_sha256'] and summary['nas_readback']['verified']
prior=json.loads(legacy.read_text());old=[r for r in prior['reports'] if r['report_period']==plan['required_periods']['last_legacy']];assert len(old)==1 and len(old[0]['equities'])==30;old=old[0]
state={'plan_sha256':sha(card/'plan.json'),'phase':'source_Recorder_load_attempted','source_Recorder_object_load_attempts':1,'legacy_result_loads_total':2,'external_HTTP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/json.loads((modern/'plan.json').read_text())['calendar_provider']),region='us',kernels=1)
with R.uri_context(summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id']);assert rec.status=='FINISHED';rows=rec.load_object('analysis/member_rows')
assert isinstance(rows,list) and len(rows)==390;periods=sorted({r['report_period'] for r in rows});assert len(periods)==13 and periods[0]==plan['required_periods']['first_NPORT']
first=[r for r in rows if r['report_period']==periods[0]];assert len(first)==30
counts={'terminal_parenthesized_removed':0,'terminal_bare_lowercase_removed':0}
def norm(name):
 s=unicodedata.normalize('NFKC',name).strip()
 while True:
  m=re.search(r'\s*\([a-z]\)\s*$',s)
  if not m:break
  s=s[:m.start()].rstrip();counts['terminal_parenthesized_removed']+=1
 m=re.search(r'\s+[a-z](?:,[a-z])*\s*$',s)
 if m:s=s[:m.start()].rstrip();counts['terminal_bare_lowercase_removed']+=1
 s=re.sub(r'[.,]',' ',s)
 return re.sub(r'\s+',' ',s).strip().casefold()
legacy_by={norm(r['security_original']):r for r in old['equities']};modern_by={norm(r['security_original']):r for r in first};assert len(legacy_by)==len(modern_by)==30
shared=sorted(set(legacy_by)&set(modern_by));matched=[{'normalized_security':n,'legacy_original_name':legacy_by[n]['security_original'],'NPORT_original_name':modern_by[n]['security_original'],'NPORT_cusip':modern_by[n]['cusip_original'],'NPORT_identifier_status':modern_by[n]['identifier_status']} for n in shared]
valid=[r for r in matched if r['NPORT_identifier_status']=='original_unique_CUSIP'];placeholder=[r for r in matched if r['NPORT_identifier_status']!='original_unique_CUSIP'];assert len({r['NPORT_cusip'] for r in valid})==len(valid)
modern_counts=dict(collections.Counter(r['identifier_status'] for r in rows))
result={'decision':'last_legacy_2019Mar_to_first_NPORT_2019Sep_identity_bridge_partial','plan_sha256':sha(card/'plan.json'),'legacy_accession':old['accession'],'legacy_period':old['report_period'],'modern_first_period':periods[0],'missing_intervening_period':plan['required_periods']['missing_intervening_report'],'modern_report_count':len(periods),'modern_member_rows':len(rows),'modern_identifier_status_counts':modern_counts,'legacy_rows':30,'modern_first_rows':30,'normalization_counts':counts,'exact_normalized_name_overlap':len(shared),'matched_unique_CUSIP':len(valid),'matched_placeholder_or_unknown_CUSIP':len(placeholder),'legacy_only_names':sorted(set(legacy_by)-set(modern_by)),'modern_only_names':sorted(set(modern_by)-set(legacy_by)),'matched_original_identity_rows':matched,'complete_2011_2026_security_identity_bridge_qualified':False,'weighted_turnover_feature_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
small={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_recorder_id':summary['recorder_id'],'source_NAS_sha256':summary['nas_evidence']['sha256'],'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['2019-03 to2019-09 is a six-month source gap, not adjacent-report turnover.','Exact matched names/CUSIPs do not solve all legacy legal aliases or modern placeholders.','No full-period signal, outcome read or funded Stage1 account.']};(card/'summary.json').write_text(json.dumps(small,indent=2)+'\n');state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({k:result[k] for k in ('legacy_period','modern_first_period','exact_normalized_name_overlap','matched_unique_CUSIP','matched_placeholder_or_unknown_CUSIP','legacy_only_names','modern_only_names')})

[49338:MainThread](2026-10-23 09:53:23,807) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[49338:MainThread](2026-10-23 09:53:23,809) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[49338:MainThread](2026-10-23 09:53:23,809) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'legacy_period': '2019-03-31', 'modern_first_period': '2019-09-30', 'exact_normalized_name_overlap': 24, 'matched_unique_CUSIP': 22, 'matched_placeholder_or_unknown_CUSIP': 2, 'legacy_only_names': ['asml holding nv nys', 'cypress semiconductor corp', 'kla-tencor corp', 'mellanox technologies ltd', 'silicon motion technology corp adr', 'taiwan semiconductor manufacturing co ltd adr'], 'modern_only_names': ['asml holding nv', 'cabot microelectronics corp', 'cirrus logic inc', 'kla corp', 'semtech corp', 'tsmc']}


# SOXX 旧表单至 N-PORT：2019身份桥部分通过

v1错把2018-12当首份 N-PORT，已原样保留失败。v2只修报告期与旧源字段：使用最后一份已核实的 **2019-03-31** 旧表单 SOXX 30股，以及原生 Qlib Recorder 中首份 **2019-09-30** N-PORT 30股。首次解析又发现旧卡字段名是 `report_period`，不是 `report_date`；在读取新 Recorder 前失败，另以 `parser_repair_plan.json` 透明增加一次旧结果重读，未改来源、时钟或证券公式。全程无市场行情、未来收益、拟合和账户。

脚注/点号的冻结机械规范化后，两份相隔六个月的原始披露精确名称对上 **24/30**；其中 **22个**对应 N-PORT 原始唯一 CUSIP，Marvell 与 NXP 的两行仍是占位 `000000000`。两侧各六个未匹配名称，包括 ASML、KLA、TSMC 的写法过渡，也可能包含真正的进出；不凭相似名字自动合并。13份 N-PORT 共390行，357行是唯一原始 CUSIP、33行是占位。完整差异见 `result.json`。

**来源桥仍非连续季度身份链。** 当前归档包中未包含 **2019-06-30** 原始 SOXX 30股报告；不能把2019-03→09的24个重合名称当作一个季度真实换手，也不能由报告市值差计算成交或外推至2011–2026。下一步先查2019-06是否有独立原始申报及首次可用时点，再核六个未匹配名与占位 CUSIP；完成全期身份/时点后才可冻结轮换特征的信息门。本卡没有证明 SOXX 费用后超额。
